# Prostate158 RQ2 — 3D Reconstruction of 2D U-Net Segmentation (Colab)

**Thesis:** *Segmentasi Zona Anatomi pada Citra MRI Pasien Kanker Prostat menggunakan 2D U-NET dan Rekonstruksi 3D.*

**RQ2:** *Bagaimana Implementasi Rekonstruksi 3D Citra MRI Pasien Kanker Prostat dari hasil Segmentasi 2D U-Net?*

This notebook is the **execution entry point** for the RQ2 pipeline that already
lives in the repository. It does **not** re-implement any reconstruction logic —
it drives `scripts/run_rq2_reconstruction.py`, which reuses `src/` (inference →
explicit-index reassembly → inverse preprocessing → original-space restoration →
geometry validation → metrics → visualization).

## Scope & research integrity (read first)

- **RQ1 is frozen.** No retraining, no architecture change, no AMP experiment, no
  new model. The existing epoch-77 checkpoint is loaded **read-only**.
- **Nothing here writes to `results/exp01_baseline`.** All RQ2 outputs go to the
  RQ2 output directory on Google Drive.
- **Two different "Dice" numbers — never conflate them:**
  - **A. Reconstruction fidelity** (`scripts/roundtrip_test.py`): GT mask →
    preprocess → reconstruct → inverse → original space gives **Dice = 1.0**,
    shape/affine exact, **with no model involved**. This only proves the
    reconstruction *transform* preserves the mask.
  - **B. Real model segmentation performance** (this notebook, Section 12):
    predicted 3D mask vs ground truth → CG/PZ Dice, HD95, ASD. **This is the
    model's accuracy.** Round-trip Dice = 1.0 is **not** model accuracy.
- **Label mapping (human-verified in 3D Slicer):** `0 = Background`,
  `1 = Central Gland (CG)`, `2 = Peripheral Zone (PZ)`. Never changed here.

**Frozen checkpoint:** `results/exp01_baseline/best_model.pt` (epoch 77, in=1, out=3).


## 1. Mount Google Drive

Final RQ2 artifacts must persist on Drive, not only under `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration (edit here)

The single editable cell. Paths follow the project's Drive layout
(`/content/drive/MyDrive/THESIS_PROSTATE158/`).

> **`TEST_DIR` is not assumed.** Point it at the directory that *directly contains*
> the 19 official held-out test case folders (the extracted Prostate158 test
> archive, DOI 10.5281/zenodo.6592345). If you are unsure of the exact case-root,
> run `scripts/audit_test_archive.py` first (Section 8 shows how).

In [ ]:
# --- Repository (cloned into Colab) ---
REPO_URL     = "https://github.com/amadeussandro/prostate-mri-segmentation-thesis.git"
REPO_BRANCH  = "main"
PROJECT_DIR  = "/content/prostate-mri-segmentation-thesis"

# --- Google Drive base ---
DRIVE_BASE   = "/content/drive/MyDrive/THESIS_PROSTATE158"

# --- Data / checkpoint / outputs ---
DATASET_DIR     = f"{DRIVE_BASE}/dataset"
# EDIT THIS to your extracted 19-case TEST case-root (dir that directly holds the case folders):
TEST_DIR        = f"{DRIVE_BASE}/dataset/test/extracted"
CHECKPOINT_PATH = f"{DRIVE_BASE}/results/exp01_baseline/best_model.pt"
OUTPUT_DIR      = f"{DRIVE_BASE}/results/exp04_rq2_reconstruction"

# --- Config used by the pipeline (preprocessing + model + official split) ---
CONFIG_PATH  = f"{PROJECT_DIR}/configs/config_baseline.yaml"

# Frozen-scope expectations (verified below, never enforced by training).
EXPECTED_EPOCH        = 77
EXPECTED_IN_CHANNELS  = 1
EXPECTED_OUT_CLASSES  = 3
EXPECTED_TEST_CASES   = 19

for k in ["REPO_BRANCH", "PROJECT_DIR", "DRIVE_BASE", "TEST_DIR", "CHECKPOINT_PATH", "OUTPUT_DIR"]:
    print(f"{k:16s} = {globals()[k]}")

## 3. Repository setup (clone or pull — no duplicates)

Clone the repo if absent, otherwise fetch + hard-checkout the latest `main`.

In [ ]:
import os, subprocess

def sh(*args, check=True):
    print("$", " ".join(args))
    return subprocess.run(args, check=check)

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    sh("git", "clone", "--branch", REPO_BRANCH, REPO_URL, PROJECT_DIR)
else:
    sh("git", "-C", PROJECT_DIR, "fetch", "origin", REPO_BRANCH)
    sh("git", "-C", PROJECT_DIR, "checkout", REPO_BRANCH)
    sh("git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", REPO_BRANCH)

os.chdir(PROJECT_DIR)
print("\ncwd:", os.getcwd())
sh("git", "-C", PROJECT_DIR, "log", "--oneline", "-1")

## 4. Install dependencies

Only what the RQ2 pipeline needs. Colab already ships PyTorch, NumPy, pandas and
matplotlib; this installs/repins the imaging + config libs from the repo's
`requirements.txt`.

In [ ]:
# torch is preinstalled on Colab GPU runtimes; do not reinstall/train.
%pip install -q nibabel>=5.4 scipy>=1.13 pyyaml>=6.0 pandas>=2.0
import torch, nibabel, numpy, scipy, pandas, matplotlib
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("nibabel", nibabel.__version__, "| numpy", numpy.__version__, "| scipy", scipy.__version__)

## 5. Verify environment (fail loudly on any missing path)

Checks the repository, dataset root, official test case-root, checkpoint, and
creates the output directory. **Stops with a clear error if anything required is
missing** — no silent fallbacks.

In [ ]:
import os, sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

problems = []
if not os.path.isfile(os.path.join(PROJECT_DIR, "scripts", "run_rq2_reconstruction.py")):
    problems.append(f"RQ2 script missing under {PROJECT_DIR} (repo not set up correctly).")
if not os.path.isdir(DRIVE_BASE):
    problems.append(f"Drive base not found: {DRIVE_BASE} (is Drive mounted / path correct?).")
if not os.path.isfile(CHECKPOINT_PATH):
    problems.append(f"Frozen checkpoint not found: {CHECKPOINT_PATH}")
if not os.path.isdir(TEST_DIR):
    problems.append(f"TEST_DIR not found: {TEST_DIR} (set it to the extracted 19-case test case-root).")

os.makedirs(OUTPUT_DIR, exist_ok=True)

if problems:
    raise FileNotFoundError("Environment check failed:\n  - " + "\n  - ".join(problems))
print("Environment OK.")
print("  checkpoint :", CHECKPOINT_PATH)
print("  test root  :", TEST_DIR)
print("  output dir :", OUTPUT_DIR)

## 6. Verify the frozen checkpoint (read-only, no training)

Loads the checkpoint, rebuilds the model from its **own embedded config**, and
confirms `in_channels = 1`, `out_channels = 3`, and (where available) epoch 77.

In [ ]:
import torch
from src.model import build_model
from src.utils import get_device

device = get_device()
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
assert "model_state_dict" in ckpt, f"Checkpoint has no model_state_dict; keys: {sorted(ckpt)}"

ckpt_cfg = ckpt.get("config", {})
model = build_model(ckpt_cfg).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

in_ch  = int(getattr(model, "in_channels", -1))
out_ch = int(getattr(model, "out_channels", -1))
epoch  = ckpt.get("epoch", "unknown")
best   = ckpt.get("best_metric", float("nan"))

print(f"architecture : {ckpt_cfg.get('model', {}).get('architecture', type(model).__name__)}")
print(f"in_channels  : {in_ch}  (expected {EXPECTED_IN_CHANNELS})")
print(f"out_channels : {out_ch}  (expected {EXPECTED_OUT_CLASSES})")
print(f"epoch        : {epoch}   (expected {EXPECTED_EPOCH})")
print(f"best_metric  : {best}  (training-loop slice-level val metric)")

assert in_ch == EXPECTED_IN_CHANNELS,  f"Expected T2W single-channel input, got in_channels={in_ch}"
assert out_ch == EXPECTED_OUT_CLASSES, f"Expected 3-class output (0=bg,1=CG,2=PZ), got {out_ch}"
if isinstance(epoch, int) and epoch != EXPECTED_EPOCH:
    print(f"WARNING: checkpoint epoch {epoch} != expected {EXPECTED_EPOCH} (continuing; verify this is the intended frozen checkpoint).")
print("\nCheckpoint verified (read-only).")

## 7. Verify the official held-out test split (19 cases)

Resolves the official split from the repo CSVs + `TEST_DIR`, asserts exactly
**19** test cases and no train/val leakage. Does not substitute another split.

*Optional deeper audit* (shapes, labels, GT presence): run
`scripts/audit_test_archive.py --extract-dir {TEST_DIR}` — read-only, no model.

In [ ]:
from src.splits import load_official_split

official = load_official_split(
    train_csv=os.path.join(PROJECT_DIR, "dataset", "prostate158_train", "train.csv"),
    valid_csv=os.path.join(PROJECT_DIR, "dataset", "prostate158_train", "valid.csv"),
    test_dir=TEST_DIR,
)
test_ids = official.test_ids or []
print(f"train: {len(official.train_ids)} | val: {len(official.val_ids)} | test: {len(test_ids)}")
print("test ids:", test_ids)
assert len(test_ids) == EXPECTED_TEST_CASES, (
    f"Expected {EXPECTED_TEST_CASES} official test cases, resolved {len(test_ids)}. "
    "Check TEST_DIR points at the case-root (dir that directly holds the 19 case folders).")
print(f"\nOfficial held-out test split OK: {len(test_ids)} cases, no leakage.")

## 8. Single-case sanity check (STOP here if it fails)

Runs the **existing** shared inference + reconstruction path
(`src.evaluate.predict_case_reconstructed`) on ONE official test case and checks:
model loads, T2W loads, preprocessing runs, 2D inference runs, prediction is
valid, 2D→3D reconstruction succeeds, original shape + affine preserved, labels
⊂ {0,1,2}. **If this fails, do not run all 19 cases.**

In [ ]:
import numpy as np
from src.dataset import ProstateZonal2DDataset
from src.evaluate import predict_case_reconstructed
from src.reconstruction import validate_reconstruction_geometry
from src.transforms import PreprocessingConfig

SANITY_OK = False
sanity_pid = test_ids[0]
preproc = PreprocessingConfig.from_dict(ckpt_cfg.get("preprocessing", {}))

ds = ProstateZonal2DDataset(
    dataset_root=TEST_DIR, patient_ids=[sanity_pid],
    image_name="t2.nii.gz", mask_name="t2_anatomy_reader1.nii.gz",
    slice_sampling="all", preprocessing_config=preproc, cache_data=True,
)
cp = predict_case_reconstructed(sanity_pid, model, ds, device)
val = validate_reconstruction_geometry(cp.pred_nii, cp.geometry, valid_labels=(0, 1, 2))

print(f"case {sanity_pid}: n_slices={len(cp.slice_indices)}  "
      f"pred labels={sorted(int(v) for v in np.unique(cp.pred_original))}")
print("geometry validation:", {k: val[k] for k in
      ["shape_match", "affine_match", "spacing_match", "orientation_match", "labels_valid", "all_ok"]})

assert val["all_ok"], f"Single-case reconstruction failed geometry validation: {val}"
assert set(np.unique(cp.pred_original)).issubset({0, 1, 2}), "Prediction has labels outside {0,1,2}"
SANITY_OK = True
print(f"\nSANITY CHECK PASSED for case {sanity_pid}. Safe to run all {len(test_ids)} cases.")

## 9. Run the full RQ2 pipeline (all 19 test cases)

Runs the **existing** `scripts/run_rq2_reconstruction.py` (no reconstruction logic
is re-implemented in this notebook). Outputs are written directly to the Drive
`OUTPUT_DIR`. Only runs if the single-case sanity check passed.

In [ ]:
assert SANITY_OK, "Refusing to run all cases: single-case sanity check did not pass (Section 8)."

import subprocess, sys
cmd = [
    sys.executable, "scripts/run_rq2_reconstruction.py",
    "--config", CONFIG_PATH,
    "--checkpoint", CHECKPOINT_PATH,
    "--test-dir", TEST_DIR,
    "--output-dir", OUTPUT_DIR,
    "--split", "test",
]
print("$", " ".join(cmd), "\n")
proc = subprocess.run(cmd)
assert proc.returncode == 0, f"RQ2 pipeline exited with code {proc.returncode}"
print("\nRQ2 pipeline finished.")

## 10. Inspect the output directory

In [ ]:
import os
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort()
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for fn in sorted(files):
        if fn == ".gitkeep":
            continue
        size = os.path.getsize(os.path.join(root, fn))
        print("  " * (level + 1) + f"{fn}  ({size/1024:.1f} KB)")

## 11. Final metrics (read from generated files — no fabricated values)

Reads `summary.json` / `metrics_summary.csv` / `reconstruction_validation.csv`.
Shows N/A for any value not present. Labels: `1 = CG`, `2 = PZ`.

> Reminder: these are **model** segmentation metrics (Section B). They are **not**
> the round-trip fidelity Dice = 1.0 (Section A / Section 12 below).

In [ ]:
import os, json
import pandas as pd

CLASS_NAME = {1: "CG (Central Gland)", 2: "PZ (Peripheral Zone)"}

def _na(x):
    return "N/A" if x is None else x

summary_json = os.path.join(OUTPUT_DIR, "summary.json")
summary_csv  = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
valid_csv    = os.path.join(OUTPUT_DIR, "reconstruction_validation.csv")

n_cases = "N/A"
if os.path.isfile(summary_json):
    meta = json.load(open(summary_json)).get("metadata", {})
    n_cases = meta.get("n_cases", "N/A")
    print("checkpoint epoch     :", _na(meta.get("checkpoint_epoch")))
    print("evaluation space     :", _na(meta.get("evaluation_space")))
    print("aggregation          :", _na(meta.get("aggregation")))
    print("label mapping status :", _na(meta.get("label_mapping_status")))
    print("recon geometry all_ok:", _na(meta.get("reconstruction_geometry_all_ok")))
print("cases processed      :", n_cases)

print("\n== Real 3D model segmentation metrics (mean +/- SD across cases) ==")
if os.path.isfile(summary_csv):
    df = pd.read_csv(summary_csv)
    for cid in (1, 2):
        row = df[(df["class_id"] == cid)]
        print(f"\n{CLASS_NAME[cid]}:")
        for metric in ["dice", "iou", "hd95_mm", "asd_mm", "precision", "recall"]:
            r = row[row["metric"] == metric]
            if len(r):
                m, s = float(r["mean"].iloc[0]), float(r["std"].iloc[0])
                print(f"  {metric:9s}: {m:.4f} +/- {s:.4f}")
            else:
                print(f"  {metric:9s}: N/A")
else:
    print("metrics_summary.csv not found -> N/A")

print("\n== Geometry validation ==")
if os.path.isfile(valid_csv):
    vdf = pd.read_csv(valid_csv)
    n_ok = int(vdf["all_ok"].astype(str).str.lower().eq("true").sum())
    print(f"  all_ok: {n_ok}/{len(vdf)} cases pass all geometry checks")
else:
    print("  reconstruction_validation.csv not found -> N/A")

## 12. Representative visualizations (generated by the pipeline)

Displays the figures the pipeline already produced under
`visualizations/` (objective good / median / challenging selection). No second
visualization implementation is created here.

**These figures show model predictions vs ground truth — not the round-trip
fidelity result.**

In [ ]:
import glob, os
from IPython.display import Image, display, Markdown

viz_root = os.path.join(OUTPUT_DIR, "visualizations")
pngs = sorted(glob.glob(os.path.join(viz_root, "**", "*.png"), recursive=True))
if not pngs:
    print("No visualization PNGs found under", viz_root)
else:
    # Prefer one 3D-projection figure per representative case, then a few axial panels.
    proj = [p for p in pngs if "3d_projections" in os.path.basename(p)]
    axial = [p for p in pngs if p not in proj]
    for p in proj + axial[:6]:
        display(Markdown(f"**{os.path.relpath(p, OUTPUT_DIR)}**"))
        display(Image(filename=p))

## 13. Final artifact listing

In [ ]:
import os, glob

def count(pattern):
    return len(glob.glob(os.path.join(OUTPUT_DIR, pattern), recursive=True))

print("RQ2 artifacts under:", OUTPUT_DIR)
print("  reconstructed NIfTI (reconstructions_3d/*.nii.gz):", count("reconstructions_3d/*.nii.gz"))
print("  2D prediction stacks (predictions_2d/*.npz)      :", count("predictions_2d/*.npz"))
print("  per-case metadata (metadata/*.json)             :", count("metadata/*.json"))
print("  visualization figures (visualizations/**/*.png) :", count("visualizations/**/*.png"))
for f in ["metrics_per_case.csv", "metrics_summary.csv",
          "reconstruction_validation.csv", "summary.json", "report.md"]:
    path = os.path.join(OUTPUT_DIR, f)
    print(f"  {f:32s}: {'present' if os.path.isfile(path) else 'MISSING'}")
print("\n  NIfTI dir     :", os.path.join(OUTPUT_DIR, "reconstructions_3d"))
print("  viz dir       :", os.path.join(OUTPUT_DIR, "visualizations"))
print("  report.md     :", os.path.join(OUTPUT_DIR, "report.md"))

## 14. Research integrity recap

- **A. Reconstruction fidelity** (`scripts/roundtrip_test.py`, and the round-trip
  unit tests): shape exact, affine exact, **round-trip Dice = 1.0** — the
  reconstruction transformation preserves the mask. **No model involved.**
- **B. Model segmentation performance** (Section 11): predicted 3D mask vs ground
  truth → CG/PZ Dice, HD95, ASD. **This is the model's real accuracy.**

**Round-trip Dice = 1.0 is NEVER the model's segmentation accuracy.**

RQ1 (model + training) remains frozen; `results/exp01_baseline` is never modified
by this notebook. All RQ2 artifacts persist on Google Drive under
`.../THESIS_PROSTATE158/results/exp04_rq2_reconstruction/`.
